# Display features

In [ ]:
import pickle

with open(r"C:\Users\bilel.guetarni\Desktop\SEQ-RT\artix.pkl", "rb") as f:
    artix = pickle.load(f)

with open(r"C:\Users\bilel.guetarni\Desktop\SEQ-RT\hnscc.pkl", "rb") as f:
    hnscc = pickle.load(f)

In [ ]:
for p in hnscc:
    for ct in p.ct:
        if ct.rtdose is None:
            print(f"ct {ct.path} has no RTDOSE")

        if ct.rtstruct is None:
            print(f"ct {ct.path} has no RTSTRUCT")

In [ ]:
p.ct[0].rtdose.path

In [ ]:
import pandas
import re

def transform_(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None

def retrieve_xero_info(p):
    cm = pandas.DataFrame(p.clinical_measurements)
    cm = cm[(cm["type"] == "AE") & (cm["sample"] == "XEROSTOMIE")]
    cm["value"] = cm["value"].apply(transform_)
    cm.loc[(cm["visitID"] == "S0"), "visitID"] = "Inclusion"
    if (cm[(cm["visitID"] == "Inclusion") & (cm["sample"] == "XEROSTOMIE")]["value"].astype('Int64') > 0).any():
        # exclude patient because xerstomia baseline present
        xerostomia = None
    else:
        try:
            cm = cm[cm["visitID"] == "M6"]
            xerostomia = int(cm["value"].values[0])
        except (IndexError, ValueError):
            xerostomia = None
    return xerostomia

xero = list(map(retrieve_xero_info, artix))
stats = {k: xero.count(k) for k in set(xero)}
print("artix:")
for j, c in stats.items():
    print(f"\t {j}: {c} ({int(100*c/len(xero))}%)")

In [ ]:
def retrieve_xero_info(p):
    try:
        xerostomia = int(p.clinical['Xerostomia Grade'])
    except ValueError:
        xerostomia = None
    return xerostomia

xero = list(map(retrieve_xero_info, hnscc))
stats = {k: xero.count(k) for k in set(xero)}
print("hnscc:")
for j, c in stats.items():
    print(f"\t {j}: {c} ({int(100*c/len(xero))}%)")

# Fit

### original
parotid_gland_ipsi
parotid_gland_contra
submandibular_gland_ipsi
submandibular_gland_contra
mandible
oral_cavity

### total segmentator
parotid_gland_left
parotid_gland_right
submandibular_gland_right
submandibular_gland_left
superior_pharyngeal_constrictor
middle_pharyngeal_constrictor
inferior_pharyngeal_constrictor
mandible
esophagus

In [ ]:
import plotly.express as px
from sklearn.decomposition import PCA

def pca_visualization(x, y):
    x = PCA(n_components=2, random_state=0).fit_transform(x)
    fig = px.scatter(x=x[:,0], y=x[:,1], color=y)
    fig.show()

# Analysis

In [ ]:
import os
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments"

for exp in os.listdir(path_):
    result_df = []
    for run in os.listdir(os.path.join(path_, exp)):
        with open(os.path.join(path_, exp, run, "params.json"), "r") as f:
            params = json.load(f)

        metrics_df = pandas.read_csv(os.path.join(path_, exp, run, "metrics.csv"))
        metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
        avg_metrics = metrics_df.mean().to_dict()
        for k, v in avg_metrics.items():
            try:
                oars = params["oars"]
            except KeyError:
                oars =  params["oar"]

            try:
                features = tuple(params["features"].items())
            except AttributeError:
                features = params["features"]
                features = {k: len(list(filter(lambda i: i[0] == k, features))) for k in set(map(lambda i: i[0], features))}
                features = tuple(features.items())
            
            result_df.append({"exp": exp, "run": run, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "oars": oars, "features": features})
    result_df = pandas.DataFrame(result_df)
    fig = px.bar(result_df, x="run", y="value", color="metric", barmode="group", hover_data=['features', 'oars'], title=exp)
    fig.update_yaxes(range=[0,1])
    # fig.update_layout(autosize=False, width=1400, height=2000)
    fig.show()

In [ ]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\005"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)

fig = px.bar(result_df, x="oar", y="value", color="metric", barmode="group")
fig.update_layout(autosize=False, width=800, height=400)
fig.show()
# fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot.png")

In [ ]:
import os, json
import pandas
import plotly.express as px


path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\002"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "run": run.split("_")[-1]})

result_df = pandas.DataFrame(result_df)

for run in result_df["run"].unique():
    print(run)
    fig = px.bar(result_df[result_df["run"] == run], x="oar", y="value", color="metric", barmode="group")
    fig.update_layout(autosize=False, width=800, height=400)
    fig.show()
    fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot_{}.png".format(run))

In [ ]:


import os, json
import pandas
import plotly.express as px

run_names = {1: "shape", 2: "first order", 3: "texture", 4: "radiomics", 5: "radiomics+dosiomics"}



path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\002"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "run": run_names[int(run.split("_")[-1])]})

result_df = pandas.DataFrame(result_df)

fig = px.bar(result_df, x="oar", y="value", facet_row="metric", barmode="group", facet_col="run")
fig.update_layout(autosize=False, width=1200, height=400)

# # Get unique x values (categories)
# x_vals = result_df["oar"].unique()

# # Add vertical dashed lines *between* categories
# for i in range(len(x_vals) - 1):
#     fig.add_vline(
#         x=i + 0.5,  # halfway between category i and i+1
#         line=dict(color="black", width=2, dash="dash"),
#         row="all", col="all"  # apply to all facet subplots
#     )

# fig.show()
fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot.svg", width=1400, height=1200)

In [ ]:
# result_df.to_csv(r"C:\Users\bilel.guetarni\Desktop\tmp\metrics.csv")
# result_df = result_df.drop(columns=["std"])
# result_df = result_df.set_index(["oar", "run"])
result_df

In [ ]:
# Pivot
table = result_df.pivot(index=["oar", "run"], columns="metric", values="value")

# Flatten the index so oar and run are normal columns
table = table.reset_index()

table.to_csv(r"C:\Users\bilel.guetarni\Desktop\tmp\metrics.csv")

In [ ]:
import os, json
import pandas
import plotly.express as px

result_df = []
path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\007"
for run in os.listdir(path_):
    # print(run)
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

        # fts = params["features"]["dvh"]
        # if isinstance(fts, list) and len(fts) == 0:
        #     continue

        # print(run)
        # print(params["oars"])
        # print(params["features"])

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "run": run, "oars": params["oars"], "features": tuple(params["features"].items())})

fig = px.bar(result_df, x="run", y="value", color="metric", facet_row="metric", barmode="group", hover_data=['features', 'oars'])
fig.update_yaxes(range=[0, 1])
fig.update_layout(autosize=False, width=1200, height=800)
fig.show()

## 008

In [ ]:
import os, glob, json
import tqdm

def filter_features(features_dict):
    return "+".join([i for i in features_dict.keys() if features_dict[i] == -1])

result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_*\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        oars = "+".join(params["oars"])
        features = filter_features(params["features"])
        if params["feature_reduction_N"]:
            dim = params["feature_reduction_N"]
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()

    for k, v in avg_metrics.items():
        result_df.append({"metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "exp": os.path.split(exp)[1],
                          "oars": oars, "features": features, "dim": dim})

In [ ]:
import pandas
import plotly.express as px

df = pandas.DataFrame(result_df)
df = df[(df["metric"] == "f1_score") & (df["dim"] == 100)]

fn_ = lambda i: 1 if "deepnn" in i else 0
df["deepnn"] = df["features"].map(fn_)

sorted_features = sorted(df["features"].unique(), key= lambda i: len(i.split("+")))
print(sorted_features)

print(df["oars"].unique())

fig = px.bar(df, x="oars", y="value", color="oars", facet_row="features", facet_row_spacing=0.01,
            barmode="group", hover_data=['features', 'oars'], category_orders={"features": sorted_features})
fig.update_yaxes(range=[0, 1])
fig.update_layout(autosize=False, width=1000, height=8000)
fig.show()

In [ ]:
import itertools
import pandas
import plotly.express as px
import os, glob, json
import tqdm

N = 100

result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        oars = params["oars"]
        if len(oars) > 1:
            continue

        features = [i for i in params["features"].keys() if params["features"][i] == -1]
        if len(features) > 1:
            continue

        if params["feature_reduction_N"]:
            # dim = params["feature_reduction_N"]
            continue
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()

    result_df.append({"auc": avg_metrics["auc"], "metrics": avg_metrics, "oars": oars, "features": features, "dim": dim})

print(len(result_df))
result_df = sorted(result_df, key=lambda i: i["auc"], reverse=True)[:N]
print(len(result_df))
result_df = [[{"metric": k, "value": v, "oars": i["oars"], "features": i["features"], "dim": i["dim"]} for k, v in i["metrics"].items()] for i in result_df]
result_df = list(itertools.chain.from_iterable(result_df))
df = pandas.DataFrame(result_df)
print(len(df))

In [ ]:
import copy

dff = copy.deepcopy(df)
dff["oars"] = dff["oars"].map(lambda i: "+".join(i))
dff["features"] = dff["features"].map(lambda i: "+".join(i))

# fig = px.bar(dff, x="oars", y="value", color="features", barmode="group", hover_data=['features', 'oars', "dim"], facet_row="metric", facet_col="dim", category_orders={"dim": sorted(dff["dim"].unique())})
# fig.update_layout(xaxis=dict(categoryorder="total descending"))
# fig.update_yaxes(range=[0, 1])
# fig.update_layout(height=2000)
# fig.show()


# dff = dff[dff["dim"] == 10]
for m in dff["metric"].unique():
    fig = px.bar(dff[dff["metric"] == m], x="oars", y="value", color="features", barmode="group", hover_data=['features', 'oars', "dim"], title=m)
    # fig.update_layout(xaxis=dict(categoryorder="total descending"))
    fig.update_xaxes(categoryarray=sorted(dff["oars"].unique()))
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(height=400)
    fig.show()

## dim

In [ ]:
import pandas
import plotly.express as px
import os, glob, json
import tqdm


result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        if len(params["oars"]) > 1:
            continue
        oars = params["oars"][0]

        features = [i for i in params["features"].keys() if params["features"][i] == -1]
        if len(features) > 1:
            continue
        features = features[0]

        if params["feature_reduction_N"]:
            dim = params["feature_reduction_N"]
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "oars": oars, "features": features, "dim": dim})

df = pandas.DataFrame(result_df)
print(len(df))

In [ ]:
for m in df["metric"].unique():
    fig = px.line(df[df["metric"] == m].sort_values("dim", axis=0), x="dim", y="value", color='features', facet_col="oars", title=m)
    # fig.update_yaxes(range=[0, 1])
    fig.update_layout(width=1500)
    fig.show()

## top 100 features OARs

In [ ]:
import itertools
import pandas
import plotly.express as px
import os, glob, json
import tqdm

N = 100

result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        oars = params["oars"]
        features = [i for i in params["features"].keys() if params["features"][i] == -1]

        if params["feature_reduction_N"]:
            dim = params["feature_reduction_N"]
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()

    result_df.append({"oars": oars, "features": features, "dim": dim, **avg_metrics})

print(len(result_df))
result_df = sorted(result_df, key=lambda i: i["auc"], reverse=True)[:N]
print(len(result_df))

In [ ]:
def count_key(df, key, col):
    count = map(lambda i: 1 if key in i else 0, [i[col] for i in df])
    return sum(count)

def get_uniques(df, col):
    return set(itertools.chain.from_iterable([i[col] for i in df]))

oars = {i: count_key(result_df, i, "oars") for i in get_uniques(result_df, "oars")}
print(oars)
features = {i: count_key(result_df, i, "features") for i in get_uniques(result_df, "features")}
print(features)

## mean dose + (radiomics,DL)

In [ ]:
import pandas
import plotly.express as px
import os, glob, json
import tqdm


result_df = []

for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)

        oars = params["oars"]
        if len(oars) > 1:
            continue

        features = [i for i in params["features"].keys() if params["features"][i]]
        if len(features) > 1:
            continue
        if not(any([i in features for i in ["radiomics", "deepnn"]])):
            continue

        if params["feature_reduction_N"]:
            continue

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    std = metrics_df.std().to_dict()
    for k, v in  metrics_df.mean().to_dict().items():
        result_df.append({"std": std[k], "metric": k, "value": v, "oars": oars, "features": features, "mean dose": "no"})

for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\010_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)

        oars = params["oars"]
        if len(oars) > 1:
            continue

        features = [i for i in params["features"].keys() if params["features"][i]]
        if not(any([i in features for i in ["radiomics", "deepnn"]])):
            continue
        features.remove("dvh")

        if params["feature_reduction_N"]:
            continue

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    std = metrics_df.std().to_dict()
    for k, v in  metrics_df.mean().to_dict().items():
        result_df.append({"std": std[k], "metric": k, "value": v, "oars": oars, "features": features, "mean dose": "yes"})

df = pandas.DataFrame(result_df)
print(df)

In [ ]:
import copy

dff = copy.deepcopy(df)
dff["oars"] = dff["oars"].map(lambda i: "+".join(i))
dff["features"] = dff["features"].map(lambda i: "+".join(i))
for m in dff["metric"].unique():
    fig = px.bar(dff[dff["metric"] == m], error_y="std", x="oars", y="value", color="mean dose", barmode="group", hover_data=['features', 'oars'], facet_col="features", title=m)
    fig.update_xaxes(categoryarray=sorted(dff["oars"].unique()))
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(height=600)
    fig.show()

## 014

In [5]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\014"



result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "internal_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "internal", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

    metrics_df = pandas.read_csv(os.path.join(path_, run, "external_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "external", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)
fig = px.bar(result_df, x="features", y="value", color="metric", barmode="group", facet_col="oar", facet_row="cohort")
fig.update_layout(autosize=False, width=1600, height=900)
fig.show()

## 015

In [4]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\015"


result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "internal_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "internal", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

    metrics_df = pandas.read_csv(os.path.join(path_, run, "external_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "external", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)
fig = px.bar(result_df, x="features", y="value", color="metric", barmode="group", facet_col="oar", facet_row="cohort")
fig.update_layout(autosize=False, width=1600, height=900)
fig.show()